In [ ]:
import arcpy

# Your center point in decimal degrees (WGS84)
center_lon = -3.571295
center_lat = 53.408176

# Define spatial references
sr_wgs84 = arcpy.SpatialReference(4326)  # WGS84
sr_ed50_utm30n = arcpy.SpatialReference(23030)  # ED50 / UTM zone 30N

# Create center point in WGS84
center_point_wgs84 = arcpy.PointGeometry(arcpy.Point(center_lon, center_lat), sr_wgs84)

# Project to ED50 UTM 30N (your layer's coordinate system)
center_point_projected = center_point_wgs84.projectAs(sr_ed50_utm30n)
center_x = center_point_projected.firstPoint.X
center_y = center_point_projected.firstPoint.Y

print(f"Projected coordinates: X={center_x}, Y={center_y}")

# Create 3m (width) x 6m (length) rectangle
# Centered on the point
coords = [
    (center_x - 1.5, center_y - 3),  # Bottom-left
    (center_x + 1.5, center_y - 3),  # Bottom-right
    (center_x + 1.5, center_y + 3),  # Top-right
    (center_x - 1.5, center_y + 3),  # Top-left
    (center_x - 1.5, center_y - 3)   # Close polygon
]

# Create polygon
array = arcpy.Array([arcpy.Point(*coord) for coord in coords])
rectangle = arcpy.Polygon(array, sr_ed50_utm30n)

# Insert into your layer
layer = "your_layer_name"  # Replace with your actual layer name
with arcpy.da.InsertCursor(layer, ["SHAPE@"]) as cursor:
    cursor.insertRow([rectangle])

print("3m x 6m rectangle created successfully!")